In [1]:
import pandas as pd
import numpy as np

In [2]:
business_interruption_file_path = "dataset/srcsc-2026-claims-business-interruption.xlsx"
claim_cargo_file_path = "dataset/srcsc-2026-claims-cargo.xlsx"
claims_equipment_failure_file_path = "dataset/srcsc-2026-claims-equipment-failure.xlsx"
claims_workers_comp_file_path = "dataset/srcsc-2026-claims-workers-comp.xlsx"

In [23]:
def ratio(df, column_name):
    """
    Cleans a column to bring values toward a 0-1 range:
    - 0 to 1: No change
    - 0 to -1: Absolute value
    - > 1: Divide by 100
    - < -1: Absolute value, then Divide by 100
    """
    # Use a local variable for readability
    col = df[column_name]
    
    conditions = [
        (col >= 0) & (col <= 1),   # Pass
        (col < 0) & (col >= -1),  # Small negative -> Absolute
        (col < -1),               # Large negative -> /100
        (col > 1)                 # Large positive -> /100
    ]

    choices = [
        col,                # Keep original
        col.abs(),          # Absolute value
        (col / 100).abs(),    # Divide by 100
        col / 100           # Divide by 100
    ]

    # Apply logic and return the modified column (or update df)
    df[column_name] = np.select(conditions, choices, default=col)
    
    return df

def clean_range(df, column_name, start, end, convert = False):
    # 1. Absolute value to fix negatives
    df[column_name] = df[column_name].abs()
    
    # 2. Constrain to the start-end range
    df[column_name] = df[column_name].clip(lower = start, upper = end)
    
    # 4. Round the values up then convert to integer type
    if convert:
        df[column_name] = df[column_name].round(0)
        df[column_name] = df[column_name].astype('Int64')
    
    return df

def clean_ghost_data(df, column_name):
    return df[column_name].str.replace(r'_\?{3}\d{4}', '', regex=True)

def clean_data(filepath, sheet_no = 0):
    """
    Unified cleaning function that handles both Sheet 1 and Sheet 2 
    by checking for column existence.
    """
    
    df = pd.read_excel(filepath, sheet_name = sheet_no)
    
    id_columns = [
        'claim_id',
        'policy_id',
        'station_id',
        'solar_system',
        'origin',
        'destination',
        'shipment_id',
        'cargo_type',
        'container_type',
        'equipment_id',
        'equipment_type',
        'worker_id',
        'occupation',
        'employment_type',
        'injury_type',
        'injury_case'
    ]
    ratio_columns = [
        'production_load',
        'supply_chain_index',
        'exposure',
        'supervision_level',
        'solar_radiation',
        'debris_density'
    ]
    range_rules = [
        # Universal column (>2 occurance)
        ('energy_backup_score', 1, 5, True),
        ('avg_crew_exp', 1, 30, False),
        ('maintenance_freq', 0, 6, False),
        ('safety_compliance', 1, 5, True),
        ('weight', (1.5*1000), (250*1000), False),
        ('route_risk', 1, 5, True),
        ('distance', 1, 100, False),
        ('transit_duration', 1, 60, False),
        ('pilot_experience', 1, 30, False),
        ('vessel_age', 1, 50, False),
        ('equipment_age', 0, 10, False),
        ('maintainance_int', 100, 5000, False),
        ('usage_intensity', 0, 24, False),
        ('experience_yrs', 0.2, 40, False),
        ('accident_history_flag', 0, 1, True),
        ('psych_stress_index', 1, 5, True),
        ('gravity_level', 0.75, 1.50, False),
        ('safety_training_index', 1, 5, True),
        ('protective_gear_quality', 1, 5, True),
        ('claim_length', 3, 1000, False),
    ]
    
    if filepath == business_interruption_file_path:
        range_rules.append(('claim_count', 0, 4, True))
        range_rules.append(('claim_amount', (28*1000), (1426*1000), False))
        
    elif filepath == claim_cargo_file_path:
        range_rules.append(('claim_count', 0, 5, True))
        range_rules.append(('claim_amount', (31*1000), (678000*1000), False))
        range_rules.append(('cargo_value', (50*1000), (680000*1000), False))
        
    elif filepath == claims_equipment_failure_file_path:
        range_rules.append(('claim_count', 0 ,3, True))
        range_rules.append(('claim_amount', (11*1000), (790*1000), False))
        
    elif filepath == claims_workers_comp_file_path:
        range_rules.append(('claim_count', 0, 2, True))
        range_rules.append(('claim_amount', 5, 170, False))
        range_rules.append(('base_salary', (20*1000), (130*1000), False))
    
    # 1. Ghost Data Cleaning for non-numerical data (Removing _???XXXX)  
    for col in id_columns:
        if col in df.columns:
            df[col] = clean_ghost_data(df, col)

    # 3. Ratio Columns (0 to 1 Logic)
    for col in ratio_columns:
        if col in df.columns:
            df = ratio(df, col)

    # 4. Range Columns (Custom Min/Max)
    # We use a list of tuples to store (column_name, start, end, convert)    
    for col, start, end, convert in range_rules:
        if col in df.columns:
            df = clean_range(df, col, start, end, convert)

    return df


In [26]:
df = clean_data(business_interruption_file_path, 0)

In [ ]:
df

,production_load,energy_backup_score,supply_chain_index,avg_crew_exp,maintenance_freq,safety_compliance,exposure,claim_count
count,99814.000000,99807.0,99827.000000,99809.000000,99810.000000,99826.0,99822.000000,99819.0
mean,0.498923,3.003076,0.497772,15.541987,2.993718,2.999379,0.498838,0.100913
std,0.288422,1.415191,0.289138,8.668005,2.003051,1.412399,0.222898,0.417687
min,0.000000,1.0,0.000000,1.000000,0.000000,1.0,0.013528,0.0
25%,0.248000,2.0,0.248000,8.000000,1.000000,2.0,0.324000,0.0
50%,0.499000,3.0,0.496000,16.000000,3.000000,3.0,0.498000,0.0
75%,0.748000,4.0,0.748000,23.000000,5.000000,4.0,0.672000,0.0
max,1.000000,5.0,1.000000,30.000000,6.000000,5.0,0.999000,4.0
